# Data preparation
Tennis download and preparation are portable. NBA uses the prepared original inputs; the archived NBA preparation chain is documented rather than claimed to be one-click. Data are not redistributed.

In [1]:
from pathlib import Path
import json, subprocess, sys, shutil
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'code/train.py').is_file(), 'Open from repository root or notebooks/'
print('Repository:', ROOT.name)
def run_command(args, cwd=None, check=True):
    """Stream subprocess output into the notebook, preserving failures."""
    proc = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line.replace(str(ROOT), '.'), end='', flush=True)
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'Command failed with exit code {code}')



Repository: GM-CDLD


In [2]:
RAW = ROOT/'work/raw/tennis'
PREP = ROOT/'work/tennis_prepared'


In [3]:
run_command([sys.executable,str(ROOT/'tools/download_tennis.py'),'--output',str(RAW)],check=True)
run_command([sys.executable,str(ROOT/'tools/prepare_tennis.py'),'--raw',str(RAW),'--output',str(PREP)],check=True)

atp_matches_doubles_2017.csv 452497 SHA256 OK


atp_matches_doubles_2018.csv 442785 SHA256 OK


atp_matches_doubles_2019.csv 461811 SHA256 OK


## Check array structure
The training input root must contain NBA/ and tennis/ with fit.npz, test.npz, meta.json, player_ids.npy. No model selection uses test labels.

In [4]:
import numpy as np
d=PREP/'data/tennis'
print(json.loads((d/'meta.json').read_text()))
for part in ['fit','test']:
    with np.load(d/f'{part}.npz') as z: print(part,{k:z[k].shape for k in z.files})

{'domain': 'tennis', 'players': 153, 'slots': 2, 'context_dim': 5, 'targets': ['win', 'ace_rate', 'df_rate', 'service_win_rate'], 'classification': True, 'scaling': {'win': {'mean': 0.0, 'std': 1.0}, 'ace_rate': {'mean': 0.0584595914080936, 'std': 0.04063441969983864}, 'df_rate': {'mean': 0.04020879586360201, 'std': 0.027210435112335965}, 'service_win_rate': {'mean': 0.6740919258067972, 'std': 0.07844829140103149}}, 'fit_rows': 1962, 'test_rows': 460, 'finite_target_rows': {'fit': {'win': 1962, 'ace_rate': 1884, 'df_rate': 1884, 'service_win_rate': 1884}, 'test': {'win': 460, 'ace_rate': 460, 'df_rate': 460, 'service_win_rate': 460}}}
fit {'a': (1962, 2), 'b': (1962, 2), 'context': (1962, 5), 'cluster': (1962,), 'row_id': (1962,), 'win': (1962,), 'ace_rate': (1962,), 'df_rate': (1962,), 'service_win_rate': (1962,), 'ace_rate_other': (1962,), 'df_rate_other': (1962,), 'service_win_rate_other': (1962,)}
test {'a': (460, 2), 'b': (460, 2), 'context': (460, 5), 'cluster': (460,), 'row_id':

In [5]:
# Stage freshly prepared tennis arrays for notebook 02.
shutil.copytree(PREP/'data/tennis', ROOT/'work/prepared/data/tennis')
print('Tennis inputs staged for discovery.')


Tennis inputs staged for discovery.
